# [Chapter 9 — Practical Issues in Fitting, §9.2] Pitfall 2: Reporting Confidence Intervals, Not Just Point Estimates

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 8, 9
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
A point estimate without uncertainty is incomplete. Demonstrates bootstrap-based 95% confidence intervals on $\hat\alpha$ and shows how interval width scales with sample size.

## What you should already know
Chapter 8 fitting notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


## Bootstrap confidence intervals

Resample the time series with replacement, refit, and report the 2.5th to 97.5th percentile of the bootstrap distribution as a 95% CI.

In [ ]:
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']

alpha_true = params['c_I'] * params['beta'] * (S / params['N'])
J_true = alpha_true * I
J_obs = np.maximum(J_true + 0.10 * J_true.max() * np.random.randn(len(t)), 0)

# Bootstrap alpha_hat from the observed data
n_boot = 1000
mask = (t > 15) & (t < 35)
n_data = mask.sum()
alpha_boot = []
for _ in range(n_boot):
    idx = np.random.randint(0, n_data, size=n_data)
    J_resample = J_obs[mask][idx]
    I_resample = I[mask][idx]
    alpha_boot.append(J_resample.mean() / I_resample.mean())
alpha_boot = np.array(alpha_boot)

ci_low, ci_med, ci_high = np.percentile(alpha_boot, [2.5, 50, 97.5])
S_avg = S[mask].mean() / params['N']
R0_low = ci_low * params['tau_R'] / S_avg
R0_med = ci_med * params['tau_R'] / S_avg
R0_high = ci_high * params['tau_R'] / S_avg

print(f"Bootstrap 95% CI on alpha_hat: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Bootstrap 95% CI on R_0:       [{R0_low:.3f}, {R0_med:.3f}, {R0_high:.3f}]")
print(f"True R_0: {true_R0}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(alpha_boot, bins=40, color=BOOK_COLORS['alpha_hat'], alpha=0.6, edgecolor='black')
ax.axvline(ci_low, color=BOOK_COLORS['negative'], ls='--', lw=1)
ax.axvline(ci_high, color=BOOK_COLORS['negative'], ls='--', lw=1, label='95% CI')
ax.axvline(alpha_true[mask].mean(), color=BOOK_COLORS['neutral'], ls='-', lw=2, label='True alpha')
ax.set_xlabel(r'$\hat\alpha$ bootstrap distribution')
ax.set_ylabel('Count')
ax.set_title('Pitfall 2: report CI, not just point estimate')
ax.legend()
plt.tight_layout()
plt.show()
